# NumPy Dojo — Block 7: AV-Specific Operations

These are the problems that distinguish a generic ML interview from a Zoox interview.

Covers: **IoU**, **Non-Maximum Suppression**, **2D/3D rotation matrices**, **point cloud transforms**.

All implemented purely in NumPy. These ops are NumPy-native — PyTorch adds nothing here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)

# Box format throughout: [x1, y1, x2, y2] where (x1,y1)=top-left, (x2,y2)=bottom-right
# All coordinates in pixels (floats)

# Simulated detector output: 20 predicted boxes + scores
boxes = np.array([
    [10, 10, 60, 60],  [12, 11, 62, 61],  [15, 12, 65, 63],  # cluster 1 — true object A
    [50, 50, 58, 58],  [51, 51, 59, 59],                     # cluster 2 — true object B
    [11, 12, 61, 62],  [13, 10, 63, 60],                     # more cluster 1
    [100,100,150,160], [102,99,152,161],  [98,101,148,159],  # cluster 3 — true object C
    [200,200,230,240], [201,198,231,238],                    # cluster 4
    [10, 10, 45, 45],  [14, 14, 50, 50],                    # overlaps with cluster 1
    [300,300,350,380], [302,302,348,378],                   # cluster 5
    [80, 80, 120,130], [82, 78,122,132],  [79, 81,119,131], # cluster 6
    [10, 10, 55, 55],                                        # another overlap with cluster 1
], dtype=float)

scores = np.array([
    0.95, 0.88, 0.72, 0.91, 0.85,
    0.80, 0.75, 0.97, 0.82, 0.79,
    0.93, 0.78, 0.65, 0.60,
    0.96, 0.87, 0.89, 0.76, 0.71, 0.55
], dtype=float)

assert len(boxes) == len(scores)
print(f'Input: {len(boxes)} boxes with scores [{scores.min():.2f}, {scores.max():.2f}]')

---
## P1 — Intersection over Union (IoU)

IoU measures overlap between two boxes:  `IoU = area(intersection) / area(union)`

1. Implement **single-pair IoU**: scalar given two boxes
2. Implement **pairwise IoU matrix**: `(N, M)` given N and M boxes — **no loops**
3. Verify: identical boxes → IoU=1.0, non-overlapping → IoU=0.0

In [ ]:
def iou_single(box_a, box_b):
    """
    box_a, box_b: [x1, y1, x2, y2]
    Returns: scalar IoU in [0, 1]
    """
    # TODO: intersection top-left = max of top-lefts, bottom-right = min of bottom-rights
    # intersection area = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    # union = area_a + area_b - intersection
    pass

def iou_matrix(boxes_a, boxes_b):
    """
    boxes_a: (N, 4), boxes_b: (M, 4)
    Returns: (N, M) IoU values — no Python loops
    """
    # TODO: broadcast to compute all intersections at once
    # boxes_a[:, None, :] is (N, 1, 4) and boxes_b[None, :, :] is (1, M, 4)
    # Intersection x1 = max(boxes_a[:,None,0], boxes_b[None,:,0]) etc.
    pass

iou_mat = iou_matrix(boxes, boxes)

In [ ]:
# --- ASSERTS ---
# Identical box
assert np.isclose(iou_single(boxes[0], boxes[0]), 1.0), 'Identical boxes should have IoU=1'

# Non-overlapping
far_box = np.array([500., 500., 600., 600.])
assert np.isclose(iou_single(boxes[0], far_box), 0.0), 'Non-overlapping should have IoU=0'

# Contained box
outer = np.array([0., 0., 100., 100.])
inner = np.array([25., 25., 75., 75.])
iou_contained = iou_single(outer, inner)
expected = (50*50) / (100*100)  # intersection = inner area, union = outer area
assert np.isclose(iou_contained, expected, atol=1e-4), f'Contained IoU: {iou_contained:.4f} vs {expected:.4f}'

# Matrix
assert iou_mat.shape == (len(boxes), len(boxes))
# Diagonal should all be 1
assert np.allclose(np.diag(iou_mat), 1.0, atol=1e-5), 'Self-IoU should be 1'
# Symmetric
assert np.allclose(iou_mat, iou_mat.T, atol=1e-5), 'IoU matrix should be symmetric'
# All values in [0, 1]
assert iou_mat.min() >= 0 and iou_mat.max() <= 1 + 1e-6

# iou_matrix should match iou_single on each pair
for i in [0, 1, 7]:
    for j in [0, 4, 10]:
        assert np.isclose(iou_mat[i, j], iou_single(boxes[i], boxes[j]), atol=1e-5), f'Matrix vs single mismatch at ({i},{j})'

print('P1 PASSED ✓')

# Visualize IoU heatmap
plt.figure(figsize=(7, 6))
plt.imshow(iou_mat, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(label='IoU')
plt.title('Pairwise IoU matrix')
plt.xlabel('Box index'); plt.ylabel('Box index')
plt.tight_layout(); plt.show()

---
## P2 — Non-Maximum Suppression (NMS)

NMS removes redundant detections. Algorithm:
1. Sort boxes by score descending
2. Greedily pick the highest-scoring box
3. Remove all other boxes with IoU > threshold (they overlap the picked box)
4. Repeat with remaining boxes

Return the **indices** of kept boxes (in original ordering).

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """
    boxes:  (N, 4)
    scores: (N,)
    Returns: list of kept indices (into original boxes array)
    """
    # TODO: sort indices by descending score
    # TODO: greedy selection loop:
    #   - pick first (highest score) remaining index → keep it
    #   - compute IoU of kept box vs all remaining
    #   - discard those with IoU > threshold
    #   - repeat
    # Hint: maintain a list of 'remaining' indices
    pass

kept_indices = nms(boxes, scores, iou_threshold=0.5)
kept_boxes  = boxes[kept_indices]
kept_scores = scores[kept_indices]

In [ ]:
# --- ASSERTS ---
assert isinstance(kept_indices, (list, np.ndarray))

# After NMS: no two kept boxes should have IoU > threshold
kept_iou = iou_matrix(kept_boxes, kept_boxes)
np.fill_diagonal(kept_iou, 0)  # ignore self-overlap
assert kept_iou.max() <= 0.5 + 1e-5, f'Some kept boxes overlap too much: max IoU={kept_iou.max():.3f}'

# Should have reduced the count significantly
assert len(kept_indices) < len(boxes), 'NMS should remove some boxes'
# But should have kept at least one per cluster
assert len(kept_indices) >= 5, f'Too aggressive: only {len(kept_indices)} boxes kept'

# Highest-score box in cluster 1 (boxes 0-6, 12-13, 19) should be kept
cluster1_indices = [0, 1, 2, 5, 6, 12, 13, 19]
cluster1_scores = scores[cluster1_indices]
best_cluster1 = cluster1_indices[cluster1_scores.argmax()]
assert best_cluster1 in kept_indices, f'Best box in cluster 1 (idx={best_cluster1}) should be kept'

print(f'P2 PASSED ✓  {len(boxes)} boxes → {len(kept_indices)} after NMS')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (b, s, title) in zip(axes, [
    (boxes, scores, f'Before NMS ({len(boxes)} boxes)'),
    (kept_boxes, kept_scores, f'After NMS ({len(kept_boxes)} boxes)')
]):
    ax.set_xlim(0, 400); ax.set_ylim(400, 0)
    ax.set_aspect('equal')
    ax.set_title(title)
    for box, score in zip(b, s):
        x1,y1,x2,y2 = box
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=1, edgecolor='tomato', facecolor='none', alpha=0.7))
        ax.text(x1, y1-2, f'{score:.2f}', fontsize=6, color='tomato')
    ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

---
## P3 — 2D and 3D Rotation Matrices

Rotation matrices appear constantly in AV: rotating camera frames, transforming LiDAR points, converting between vehicle and world coordinates.

1. Build a **2D rotation matrix** R(θ) — rotates points by θ radians CCW
2. Rotate a set of 2D points
3. Build **3D rotation matrices** Rx, Ry, Rz for rotation around each axis
4. Compose rotations: R_total = Rz @ Ry @ Rx (apply x-rotation first)
5. Verify: R is orthogonal (`R @ R.T = I`), det(R) = 1

In [ ]:
def rot2d(theta):
    """
    theta: angle in radians (CCW positive)
    Returns: (2, 2) rotation matrix
    """
    # TODO: [[cos, -sin], [sin, cos]]
    pass

def rotate_points_2d(points, theta):
    """
    points: (N, 2)
    Returns (N, 2) rotated points
    """
    # TODO: R @ each point — but don't loop; use matrix multiply with transposition
    pass

def rotx(theta):
    """3D rotation around X-axis. Returns (3,3)"""
    # TODO: [[1,0,0],[0,c,-s],[0,s,c]]
    pass

def roty(theta):
    """3D rotation around Y-axis. Returns (3,3)"""
    # TODO: [[c,0,s],[0,1,0],[-s,0,c]]
    pass

def rotz(theta):
    """3D rotation around Z-axis. Returns (3,3)"""
    # TODO: [[c,-s,0],[s,c,0],[0,0,1]]
    pass

def compose_rotations(roll, pitch, yaw):
    """
    roll=Rx, pitch=Ry, yaw=Rz
    Apply order: X first, then Y, then Z
    Returns (3, 3)
    """
    # TODO: R = Rz @ Ry @ Rx
    pass

In [ ]:
# --- ASSERTS ---

# 2D: 90° CCW rotation of (1, 0) should give (0, 1)
R90 = rot2d(np.pi / 2)
rotated = R90 @ np.array([1., 0.])
assert np.allclose(rotated, [0., 1.], atol=1e-6), f'90° CCW of (1,0): {rotated}'

# 2D: 360° rotation should return to original
R360 = rot2d(2 * np.pi)
assert np.allclose(R360, np.eye(2), atol=1e-6), '360° rotation should be identity'

# 2D: orthogonality
R45 = rot2d(np.pi / 4)
assert np.allclose(R45 @ R45.T, np.eye(2), atol=1e-6), '2D rotation must be orthogonal'
assert np.isclose(np.linalg.det(R45), 1.0, atol=1e-6), '2D rotation det must be 1'

# Rotate a batch of points
pts = np.array([[1., 0.], [0., 1.], [-1., 0.], [0., -1.]])
pts_rot = rotate_points_2d(pts, np.pi / 2)
assert pts_rot.shape == (4, 2)
assert np.allclose(pts_rot[0], [0., 1.], atol=1e-6)  # (1,0) → (0,1)
assert np.allclose(pts_rot[1], [-1., 0.], atol=1e-6)  # (0,1) → (-1,0)

# 3D rotation matrices: orthogonality and det=1
for angle in [0.1, np.pi/4, np.pi/2, np.pi]:
    for R_fn in [rotx, roty, rotz]:
        R = R_fn(angle)
        assert R.shape == (3, 3)
        assert np.allclose(R @ R.T, np.eye(3), atol=1e-6), f'{R_fn.__name__}({angle:.2f}) not orthogonal'
        assert np.isclose(np.linalg.det(R), 1.0, atol=1e-6)

# Composed rotation
R_total = compose_rotations(0.1, 0.2, 0.3)
assert R_total.shape == (3, 3)
assert np.allclose(R_total @ R_total.T, np.eye(3), atol=1e-6), 'Composed rotation not orthogonal'
assert np.isclose(np.linalg.det(R_total), 1.0, atol=1e-6)

# Non-commutativity: Rx @ Ry != Ry @ Rx
assert not np.allclose(rotx(0.3) @ roty(0.3), roty(0.3) @ rotx(0.3)), 'Rotations should not commute'

print('P3 PASSED ✓')

# Visualize 2D rotation
np.random.seed(0)
pts_cloud = np.random.randn(50, 2)
angles = [0, np.pi/6, np.pi/3, np.pi/2]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, theta in zip(axes, angles):
    pts_r = rotate_points_2d(pts_cloud, theta)
    ax.scatter(pts_r[:, 0], pts_r[:, 1], s=15, alpha=0.7)
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title(f'θ = {np.degrees(theta):.0f}°')
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
plt.tight_layout(); plt.show()

---
## P4 — Point Cloud Transforms

A LiDAR scan produces a point cloud: N 3D points, each `(x, y, z)`.
Transforming point clouds is a core AV operation.

1. Apply a **rigid body transform** (rotation + translation) to a point cloud
2. Convert to **homogeneous coordinates** and apply a 4×4 transform matrix
3. **Voxelization**: bin 3D points into a grid of fixed cell size, count points per voxel
4. **Nearest neighbor in 3D**: given a query point, find the k closest points in the cloud
5. **Range filter**: keep only points within a bounding box `[x_min, x_max] × [y_min, y_max] × [z_min, z_max]`

In [ ]:
# Simulated LiDAR scan: N=2000 points in 3D, rough road-level scene
N_pts = 2000
pc = np.random.randn(N_pts, 3) * np.array([10., 10., 2.])  # spread: wide, tall, thin in Z
pc[:, 2] += 1.5  # center Z at 1.5m (above ground)

def rigid_transform(points, R, t):
    """
    points: (N, 3)
    R: (3, 3) rotation matrix
    t: (3,) translation vector
    Returns (N, 3)
    """
    # TODO: R @ each point + t — use matrix multiply, not loop
    # Hint: (N,3) @ R.T has the right shape
    pass

def transform_homogeneous(points, T):
    """
    points: (N, 3)
    T: (4, 4) homogeneous transform matrix [R|t; 0 1]
    Returns (N, 3) — converts to homogeneous, applies T, converts back
    """
    # TODO: append column of ones → (N, 4)
    # Apply: pts_h @ T.T → (N, 4)
    # Drop last column
    pass

def voxelize(points, voxel_size=1.0):
    """
    points: (N, 3)
    Returns: voxel_indices (N, 3) integer, counts dict {(i,j,k): count}
    Each point assigned to voxel floor(p / voxel_size)
    """
    # TODO: divide by voxel_size, floor to int
    # Count per voxel using np.unique on the integer indices
    pass

def knn_3d(query, points, k):
    """
    query: (3,) single point
    points: (N, 3)
    Returns indices of k nearest neighbors in points (shape: (k,))
    """
    # TODO: L2 distance from query to all points, argsort, take first k
    pass

def range_filter(points, x_range, y_range, z_range):
    """
    points: (N, 3)
    x_range, y_range, z_range: (min, max) tuples
    Returns filtered points (M, 3) where M <= N
    """
    # TODO: boolean mask per axis, combine with &
    pass

In [ ]:
# --- ASSERTS ---

# Rigid transform
R_test = rotz(np.pi / 4)
t_test = np.array([1., 2., 3.])
pc_transformed = rigid_transform(pc, R_test, t_test)
assert pc_transformed.shape == pc.shape
# Check one point manually
pt0_expected = R_test @ pc[0] + t_test
assert np.allclose(pc_transformed[0], pt0_expected, atol=1e-6)
# Rigid transform preserves pairwise distances
d_before = np.linalg.norm(pc[0] - pc[1])
d_after  = np.linalg.norm(pc_transformed[0] - pc_transformed[1])
assert np.isclose(d_before, d_after, atol=1e-5), 'Rigid transform should preserve distances'

# Homogeneous transform
T = np.eye(4)
T[:3, :3] = R_test
T[:3,  3] = t_test
pc_hom = transform_homogeneous(pc, T)
assert pc_hom.shape == pc.shape
assert np.allclose(pc_hom, pc_transformed, atol=1e-6), 'Homogeneous and rigid should agree'

# Voxelization
voxel_idx, voxel_counts = voxelize(pc, voxel_size=2.0)
assert voxel_idx.shape == (N_pts, 3)
assert sum(voxel_counts.values()) == N_pts, 'All points must be counted'

# KNN
query = np.array([0., 0., 1.5])
nn_idx = knn_3d(query, pc, k=5)
assert nn_idx.shape == (5,)
assert len(set(nn_idx)) == 5, 'KNN should return distinct indices'
# Verify they are actually closest
all_dists = np.linalg.norm(pc - query, axis=1)
true_k_nearest = np.argsort(all_dists)[:5]
assert set(nn_idx) == set(true_k_nearest), 'KNN indices should match true nearest'

# Range filter
filtered = range_filter(pc, x_range=(-5,5), y_range=(-5,5), z_range=(0,3))
assert filtered.ndim == 2 and filtered.shape[1] == 3
assert len(filtered) < N_pts, 'Filter should remove some points'
assert np.all(filtered[:, 0] >= -5) and np.all(filtered[:, 0] <= 5)
assert np.all(filtered[:, 2] >= 0)  and np.all(filtered[:, 2] <= 3)

print(f'P4 PASSED ✓')
print(f'  Voxels occupied: {len(voxel_counts)}')
print(f'  Points after range filter: {len(filtered)}/{N_pts}')

# Visualize: top-down view before/after filter
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(pc[:, 0], pc[:, 1], s=2, alpha=0.3, c=pc[:, 2], cmap='viridis')
axes[0].set_title(f'Full point cloud (N={N_pts})')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
rect = patches.Rectangle((-5,-5), 10, 10, linewidth=1.5, edgecolor='red', facecolor='none')
axes[0].add_patch(rect)

axes[1].scatter(filtered[:, 0], filtered[:, 1], s=4, alpha=0.5, c=filtered[:, 2], cmap='viridis')
axes[1].set_title(f'After range filter (N={len(filtered)})')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')

# Highlight KNN result
axes[1].scatter(*query[:2], marker='*', s=300, c='red', label='query')
for idx in nn_idx:
    if (-5 <= pc[idx,0] <= 5) and (-5 <= pc[idx,1] <= 5):
        axes[1].scatter(*pc[idx,:2], marker='o', s=60, c='orange', zorder=5)
axes[1].legend()
plt.tight_layout(); plt.show()

---
## P5 — Putting It Together: Bird's Eye View Grid

In AV perception, LiDAR points are often projected to a Bird's Eye View (BEV) grid for 2D object detection.

1. Apply a rigid transform to the point cloud (simulate vehicle motion between frames)
2. Range-filter to a 40m × 40m region around the vehicle
3. Voxelize to a 0.2m grid → gives a 200×200 BEV map
4. Compute per-cell **max height** and **point density** as two feature channels
5. Visualize as a 2-channel image

In [ ]:
# Larger, more realistic point cloud
N_large = 10000
pc_large = np.random.randn(N_large, 3) * np.array([15., 15., 1.5])
pc_large[:, 2] += 0.5  # mostly near ground

def bev_feature_map(points, x_range=(-20, 20), y_range=(-20, 20), voxel_size=0.2):
    """
    points: (N, 3)
    Returns:
        density_map: (H, W) — count of points per cell
        height_map:  (H, W) — max Z per cell (0 if no points)
    where H = W = (range / voxel_size)
    """
    x_min, x_max = x_range
    y_min, y_max = y_range
    grid_size = int((x_max - x_min) / voxel_size)
    H, W = grid_size, grid_size
    
    # Step 1: TODO — range filter (use your range_filter from P4)
    
    # Step 2: TODO — compute (i, j) grid indices for each point
    # i = int((x - x_min) / voxel_size), j = int((y - y_min) / voxel_size)
    # Clip to [0, grid_size - 1] to handle boundary
    
    # Step 3: TODO — fill density_map using np.add.at or np.bincount on flattened index
    # flat_idx = i * W + j
    
    # Step 4: TODO — fill height_map: for each cell, max Z of its points
    # Hint: iterate over unique cells or use np.maximum.at
    
    density_map = np.zeros((H, W))
    height_map  = np.zeros((H, W))
    
    return density_map, height_map

density, height = bev_feature_map(pc_large)

In [ ]:
# --- ASSERTS ---
assert density.shape == (200, 200), f'Expected (200,200), got {density.shape}'
assert height.shape  == (200, 200)
assert density.sum() > 0, 'Should have some points in range'
assert density.min() >= 0

# Total points in density map <= N_large (some may be out of range)
pts_in_range = range_filter(pc_large, (-20,20), (-20,20), (-np.inf, np.inf))
assert np.isclose(density.sum(), len(pts_in_range), atol=1), \
    f'Density total {density.sum()} should match in-range count {len(pts_in_range)}'

# Height map: cells with no points should be 0
assert height[density == 0].max() == 0

print(f'P5 PASSED ✓')
print(f'  Grid: {density.shape}, points in range: {int(density.sum())}/{N_large}')
print(f'  Occupied cells: {int((density > 0).sum())} / {200*200}')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
im0 = axes[0].imshow(density, origin='lower', cmap='Blues')
plt.colorbar(im0, ax=axes[0], label='Point count')
axes[0].set_title('BEV Density Map')

im1 = axes[1].imshow(height, origin='lower', cmap='YlOrRd')
plt.colorbar(im1, ax=axes[1], label='Max height (m)')
axes[1].set_title('BEV Height Map')

for ax in axes:
    ax.set_xlabel('X cell'); ax.set_ylabel('Y cell')
plt.suptitle('Bird\'s Eye View Feature Maps')
plt.tight_layout(); plt.show()